# PAC-Bayes Bounds with IVON

Here, we show how IVON can be used to obtain non-vacuous upper bounds on the generalization error of a neural network, in a setting where there many more parameters than there are training examples.  

We also see in this notebook how the bound explains a phenomenon that is perceived as 'mysterious' in deep learning, that is, how neural networks can fit training data with random labels: https://arxiv.org/abs/1611.03530, https://dl.acm.org/doi/10.1145/3446776

In [61]:
import math, torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
from ivonwithprior import IVON

class_a, class_b = 3, 8  # bag vs dress on FashionMNIST
datapercent = 1.0

batch_size = 20 # batch size
steps = 5000 # training steps
mc_eval = 50  # number of samples to evaluate expectation

prior_var = 0.005 # prior is an isotropic gaussian with prior_var variance
delta = 0.01 # probability of the bound
C = 1.0 # 0-1 loss is bounded by 1

Load the dataset.

In [ ]:
def binary_fashionmnist(labelnoise=0.0):
    ds = datasets.FashionMNIST(
        "../data",
        train=True,
        download=True,
        transform=transforms.ToTensor(),
    )

    idx = ((ds.targets == class_a) | (ds.targets == class_b)).nonzero().squeeze()
    X = torch.stack([ds[i][0] for i in idx])
    y = torch.tensor(
        [1.0 if ds.targets[i] == class_b else -1.0 for i in idx],
        dtype=torch.float32
    ).view(-1, 1)

    if labelnoise > 0.0:
        flip = torch.randperm(len(y))[:int(labelnoise * len(y))]
        y[flip] = -y[flip]

    if datapercent < 1:
        keep = torch.randperm(len(y))[:int(datapercent * len(y))]
        X, y = X[keep], y[keep]

    print(f"{ds.classes[class_a]} (-1) vs {ds.classes[class_b]} (1), N={len(y)}, noise={labelnoise:.0%}")

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    return X, y, loader

Minimize the bound with IVON.

In [72]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28, 500),
    nn.GELU(),
    nn.Linear(500, 300),
    nn.GELU(),
    nn.Linear(300, 1),
    nn.Tanh()
)

# logistic loss which upper bounds the 0-1 loss
class LogisticLoss(nn.Module):
    def forward(self, y_pred, y_true):
        loss = torch.log1p(torch.exp(-y_pred * y_true)) / math.log(2.0)
        return loss.mean()

X, y, loader = binary_fashionmnist(labelnoise=0.0)
N = len(X) 
num_params = sum(p.numel() for p in model.parameters())
print(f'Number of parameters: {num_params}')

opt = IVON(
    model.parameters(),
    lr=1e-3,
    ess=5000,
    hess_init=0,
    betas=(0.9, 0.9998)
)

opt.prior = model, 1 / prior_var
opt.posterior = model, 1 / prior_var

loss_fn = LogisticLoss()
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)

loader_iter = iter(loader)

for it in range(steps + 1):
    try:
        xb, yb = next(loader_iter)
    except StopIteration:
        loader_iter = iter(loader)
        xb, yb = next(loader_iter)

    with opt.sampled_params(train=True):
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()

    if it % 500 == 0: 
        model.eval()
        with opt.sampled_params(train=False):
            pred = torch.sign(model(X))
            err = (pred != y).float().mean()
        
        print(f'Steps {it:4d}: train_error={err*100:.1f}%, kl={opt.kl():.2f}')
        model.train()

    opt.step()
    sched.step()

Dress (0) vs Bag (1), N=12000, noise=0%
Number of parameters: 543101
Steps    0: train_error=40.0%, kl=0.00
Steps  500: train_error=3.1%, kl=412.21
Steps 1000: train_error=2.8%, kl=391.64
Steps 1500: train_error=1.7%, kl=371.90
Steps 2000: train_error=2.3%, kl=364.01
Steps 2500: train_error=2.1%, kl=354.73
Steps 3000: train_error=2.0%, kl=345.96
Steps 3500: train_error=9.3%, kl=345.50
Steps 4000: train_error=1.4%, kl=347.59
Steps 4500: train_error=2.1%, kl=343.74
Steps 5000: train_error=2.4%, kl=343.66


Evaluate the bound.

In [73]:
model.eval()
errs = [] 
for _ in range(mc_eval):
    with opt.sampled_params(train=False):
        pred = torch.sign(model(X))
        errs.append((pred != y).float().mean())

risk = torch.stack(errs).mean()
kl = opt.kl()

best_bound = None 
lamgrid = torch.linspace(0.05, 2.0, 20)
for lam in lamgrid: 
    bound = (
        risk
        + (kl + math.log(len(lamgrid) / delta)) / (N * lam)
        + lam * (C**2) / 8
    )
    best_bound = bound if best_bound is None else min(bound, best_bound)

print(f'Bound on generalization error: {best_bound*100:.1f}%')

Bound on generalization error: 14.8%
